# unity-lab — v0 SYSTEM PROTOTYPE (rough; hardening later)

The workstation loop in one notebook: **boot pinned Unity editor → activate license → sync a project from Drive → build (Linux or WebGL) → publish artifacts to Drive.**

- VM is **disposable**: nothing here is permanent. Source of truth = your Drive folders + the agent's repo (wired later).
- Editor tarballs are **cached on Drive** after first fetch, so later boots skip the 4.5 GB CDN download.
- License auto-activates per fresh VM (Unity Personal entitlement); ~monthly re-check.

## Before you run
1. Runtime > Change runtime type > **CPU**.
2. GATE "license" cell: fill `UNAME` / `UPASS` placeholders **locally** (never paste into chat).
3. Run all. Drive popup once (cache + project + artifacts).
4. Optional Drive folders (auto-created where possible):
   - `MyDrive/unity-lab/cache/` — editor tarballs land here on first fetch
   - `MyDrive/unity-lab/projects/<name>/Assets/...` — put a Unity project here to build it (else a demo project is scaffolded)
   - `MyDrive/unity-lab/builds/<timestamp>/` — build outputs land here

The demo project auto-authors **Cube Dodger** — a code-only endless runner (GameAuthor pattern: scripts written by the cell, game boots in the player at runtime). Play built WebGL via the tunnel cell the agent gives you.

Cells are self-sufficient and print PASS/FAIL + MEASURE lines. Rough intent: correctness first, polish later.


### 🧰 1 · Setup
**What this does:** installs tools, mounts Drive, shows the run form (cache / sync / build target). Pick CPU runtime first.

In [ ]:
#@title unity-lab v0 — init (runtime must be CPU) { display-mode: "form" }
ENABLE_DRIVE_CACHE = True #@param {type:"boolean"}
SYNC_PROJECT = True #@param {type:"boolean"}
BUILD_TARGET = "WebGL" #@param ['Linux', 'WebGL']
KEEP_ARTIFACTS_ALSO_LOCAL = True #@param {type:"boolean"}

# --- init: tools + Drive + cache dir ---
import subprocess
r = subprocess.run(['bash', '-lc', 'apt-get -qq update >/dev/null 2>&1; apt-get install -y -qq aria2 pigz xz-utils time zip >/dev/null 2>&1; echo APT_EXIT=$?'], capture_output=True, text=True)
print('apt:', (r.stdout or '').strip(), (r.stderr or '').strip()[-500:])
assert 'APT_EXIT=0' in r.stdout, 'apt failed'
r = subprocess.run(['bash', '-lc', 'echo == VM spec ==; nproc; free -m | head -2; df -h /content | tail -1; head -1 /etc/os-release'])
ALSA = subprocess.run(['bash', '-lc',
    'mkdir -p /tmp/xdg && chmod 700 /tmp/xdg; '
    'printf "pcm.!default {\n  type null\n}\nctl.!default {\n  type null\n}\n" > /etc/asound.conf; echo ALSA_XDG_OK'],
    capture_output=True, text=True)
print('audio/headless fix:', ALSA.stdout.strip())
if ENABLE_DRIVE_CACHE or SYNC_PROJECT:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    for sub in ['cache', 'projects', 'builds']:
        os.makedirs('/content/drive/MyDrive/unity-lab/' + sub, exist_ok=True)
    print('DRIVE_READY: MyDrive/unity-lab/{cache,projects,builds} ensured')
print('FORM:', 'ENABLE_DRIVE_CACHE=%s' % ENABLE_DRIVE_CACHE, '| SYNC_PROJECT=%s' % SYNC_PROJECT, '| BUILD_TARGET=%s' % BUILD_TARGET)


### 📌 2 · Config
**What this does:** writes the pinned-Unity config JSON that every later cell reads back.

In [ ]:
import json, os
# --- config: pinned editor + cache paths (derived from the form globals of cell 1) ---
CFG = {
    'VERSION': '6000.3.10f1',
    'REV': 'e35f0c77bd8e',
    'CMD5': 'd575da705f9f8d9d6bc4bf70546eca04',
    'EDITOR_URL': 'https://download.unity3d.com/download_unity/e35f0c77bd8e/LinuxEditorInstaller/Unity.tar.xz',
    'WEBGL_URL': 'https://download.unity3d.com/download_unity/e35f0c77bd8e/LinuxEditorTargetInstaller/UnitySetup-WebGL-Support-for-Editor-6000.3.10f1.tar.xz',
    'WEBGL_MD5': '74706376db4dd2202532a0d71824a793',
    'ENABLE_DRIVE_CACHE': bool(globals().get('ENABLE_DRIVE_CACHE', True)),
    'SYNC_PROJECT': bool(globals().get('SYNC_PROJECT', True)),
    'BUILD_TARGET': globals().get('BUILD_TARGET', 'WebGL'),
    'DRIVE_CACHE': '/content/drive/MyDrive/unity-lab/cache',
    'DRIVE_PROJECTS': '/content/drive/MyDrive/unity-lab/projects',
    'DRIVE_BUILDS': '/content/drive/MyDrive/unity-lab/builds',
}
json.dump(CFG, open('/content/unitylab_config.json', 'w'))
print('Pinned:', CFG['VERSION'], '| rev:', CFG['REV'], '| target:', CFG['BUILD_TARGET'], '| cache:', CFG['ENABLE_DRIVE_CACHE'])


### 🔄 3 · Keep-alive
**What this does:** best-effort anti-tab-sleep: silent audio + heartbeat. Keep the tab open during long builds.

In [ ]:
from IPython.display import Javascript, display
# --- keep-alive (best-effort: silent WebAudio + heartbeat) ---
display(Javascript('''
(function(){
  if (window.__unitylab_ka) { console.log('keepalive already running'); return; }
  window.__unitylab_ka = true;
  try {
    var ctx = new (window.AudioContext || window.webkitAudioContext)();
    var o = ctx.createOscillator(); var g = ctx.createGain();
    g.gain.value = 0.0; o.connect(g); g.connect(ctx.destination); o.start();
    if (ctx.state === 'suspended' && ctx.resume) { ctx.resume(); }
  } catch(e) { console.log('keepalive audio unavailable:', e); }
  setInterval(function(){ console.log('unity-lab keepalive ' + new Date().toISOString()); }, 60000);
  console.log('keepalive started');
})();
'''))
print('KEEPALIVE_DISPLAYED (tab must still stay open)')


### 🏇 4 · Boot editor
**What this does:** boots the pinned Unity editor: Drive-cached tarball first (md5-checked), CDN fallback with stash-back-to-cache. Skips if already extracted.

In [ ]:
import subprocess, os, time, glob, json, shutil
# --- boot pinned editor from Drive cache (fallback CDN) --- PROVEN pipeline + cache-first ---
cfg = json.load(open('/content/unitylab_config.json'))
t0 = time.time()
TGZ = '/content/Unity.tar.xz'

def checksum(path):
    import hashlib
    m = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            m.update(chunk)
    return m.hexdigest()

need_download = True
if os.path.exists(TGZ) and checksum(TGZ) == cfg['CMD5']:
    print('LOCAL TARBALL OK (md5) — reuse')
    need_download = False
cache_tgz = os.path.join(cfg['DRIVE_CACHE'], 'Unity.tar.xz') if cfg['ENABLE_DRIVE_CACHE'] else None
if need_download and cache_tgz and os.path.exists(cache_tgz):
    print('Drive cache HIT — copying 4.5 GB (measuring throughput) ...')
    tc = time.time(); sz = os.path.getsize(cache_tgz)
    subprocess.run(['bash', '-lc', 'cp "%s" /content/Unity.tar.xz' % cache_tgz])
    dt = max(time.time() - tc, 0.001)
    print('MEASURE: drive-copy %d MB in %.1f s = %.1f MB/s' % (sz / 1048576.0, dt, (sz / 1048576.0) / dt))
    if checksum(TGZ) == cfg['CMD5']:
        print('CACHE TARBALL VERIFIED — no CDN needed')
        need_download = False
    else:
        print('CACHE TARBALL CORRUPT — falling back to CDN'); os.remove(TGZ)
if need_download:
    r = subprocess.run(['bash', '-lc', 'aria2c -x8 -s8 --console-log-level=warn -d /content -o Unity.tar.xz "%s"' % cfg['EDITOR_URL']],
                       capture_output=True, text=True, timeout=1800)
    print('CDN download rc:', r.returncode)
    if (r.stdout or '').strip(): print((r.stdout or '')[-600:])
    if r.returncode != 0: print((r.stderr or '')[-2000:])
    ok = checksum(TGZ) == cfg['CMD5'] if os.path.exists(TGZ) else False
    assert ok, 'MD5_MISMATCH after download'
    if cache_tgz:
        try:
            shutil.copyfile(TGZ, cache_tgz)
            print('STASHED tarball to Drive cache for future boots')
        except Exception as e:
            print('cache stash best-effort failed (fine):', repr(e))

ubin = '/content/unity_bin.txt'
already = False
if os.path.exists(ubin):
    p = open(ubin).read().strip()
    if p and os.path.exists(p):
        miss = subprocess.run(['bash', '-lc', 'ldd "%s" | grep "not found" || true' % p], capture_output=True, text=True).stdout.strip()
        already = (not miss)
if already:
    print('GATE 1 SKIP — editor already extracted at:', p)
else:
    te = time.time()
    r = subprocess.run(['bash', '-lc', 'tar -I "xz -T0" -xf /content/Unity.tar.xz -C /content/ue || (mkdir -p /content/ue && tar -xJf /content/Unity.tar.xz -C /content/ue); echo TAR_RC=$?'],
                       capture_output=True, text=True, timeout=3600)
    print('MEASURE: extract %.1f min (xz -T0 attempt, fallback xJf)' % ((time.time() - te) / 60))
    if r.returncode != 0: print((r.stderr or '')[-2000:])
    assert r.returncode == 0, 'extract FAILED'
    cands = glob.glob('/content/ue/*/Editor/Unity') + glob.glob('/content/ue/Editor/Unity')
    assert cands, 'editor binary not found after extract'
    UNITY_BIN = cands[0]
    open(ubin, 'w').write(UNITY_BIN)
    print('editor: %.1f min total to boot' % ((time.time() - t0) / 60))
UNITY_BIN = open(ubin).read().strip()
# --- editor runtime dependencies (fresh VMs: Unity cannot exec without X libs) ---
DEPS = 'libx11-6 libx11-xcb1 libxext6 libxrandr2 libxinerama1 libxcursor1 libxi6 libfontconfig1 libglib2.0-0 libnss3 libdbus-1-3 libxcomposite1 libxdamage1 libxfixes3 libxss1 libxtst6 libdrm2 libgbm1 libgtk-3-0 libglu1-mesa libsm6 libice6 libpng16-16'
T64 = 'libasound2t64 libatk1.0-0t64 libatk-bridge2.0-0t64 libcups2t64'
apt = subprocess.run(['bash', '-lc', 'apt-get update -qq && (apt-get install -y -qq ' + DEPS + ' ' + T64 + ' || apt-get install -y -qq ' + DEPS + ' libasound2 libatk1.0-0 libatk-bridge2.0-0 libcups2)'],
                     capture_output=True, text=True, timeout=900)
print('deps install rc:', apt.returncode)
if apt.returncode != 0:
    print('--- APT STDERR (tail) ---'); print((apt.stderr or '(empty)')[-2000:])
miss = subprocess.run(['bash', '-lc', 'ldd "%s" | grep "not found" || true' % UNITY_BIN], capture_output=True, text=True).stdout.strip()
print('loader-missing libs:', miss if miss else '(none)')
assert not miss, 'still missing shared libs: ' + miss
print('GATE 1 PASS — binary:', UNITY_BIN)
subprocess.run(['bash', '-lc', '"%s" -version' % UNITY_BIN])


### 🔑 5 · License
**What this does:** one-shot Unity Personal activation — fill UNAME/UPASS placeholders first; auto-skips if this VM is already licensed.

In [ ]:
import subprocess, os
# --- license ACTIVATION --- (proven recipe; SKIPS silently if this VM is already licensed) ---
# Credentials are prompted ONLY when activation is actually needed — never stored in the notebook, never pasted into chat.
# If output shows code 132.182: wait 24h before any retry (retries extend the block).

UNITY = open('/content/unity_bin.txt').read().strip()
UCLIENT = UNITY.replace('/Editor/Unity', '') + '/Editor/Data/Resources/Licensing/Client/Unity.Licensing.Client'
assert os.path.exists(UCLIENT), 'licensing client missing: ' + UCLIENT

def smoke():
    r = subprocess.run([UNITY, '-batchmode', '-quit', '-nographics', '-logFile', '/content/lic_smoke.log'],
                       capture_output=True, text=True, timeout=900)
    liclog = open('/content/lic_smoke.log', errors='ignore').read() if os.path.exists('/content/lic_smoke.log') else ''
    BAD = [s for s in ['no valid Unity Editor license', 'was not found', 'entitlement'] if s.lower() in liclog.lower()]
    return r.returncode, BAD, liclog

skip = False
lic_xml = os.path.expanduser('~/.config/unity3d/Unity/licenses/UnityEntitlementLicense.xml')
if os.path.exists(lic_xml):
    rc0, bad0, _ = smoke()
    if rc0 == 0:
        skip = True
        print('GATE 2 SKIP — license already active on this VM (smoke rc 0)')

if not skip:
    import getpass
    UNAME = input('Unity ID email: ')
    UPASS = getpass.getpass('Unity ID password (hidden input): ')
    rc1 = subprocess.run([UCLIENT, '--show-seats', '--username', UNAME, '--password', UPASS], capture_output=True, text=True, timeout=600)
    print('SHOW_SEATS rc:', rc1.returncode); print(((rc1.stdout or '') + (rc1.stderr or ''))[-1500:])
    assert rc1.returncode == 0, 'CREDENTIAL_PRECHECK_FAILED (143.002 = wrong password; 132.182 = blocked, wait 24h)'
    rc2 = subprocess.run([UCLIENT, '--activate-all', '--include-personal', '--username', UNAME, '--password', UPASS], capture_output=True, text=True, timeout=600)
    print('ACTIVATE rc:', rc2.returncode); print(((rc2.stdout or '') + (rc2.stderr or ''))[-1500:])
    assert rc2.returncode == 0, 'ACTIVATION FAILED'

rc3, bad, liclog = smoke()
print('EDITOR_SMOKE rc:', rc3, '| bad keywords:', bad or 'none')
assert rc3 == 0, 'GATE 2 FAIL - smoke exit %d\n%s' % (rc3, '\n'.join(liclog.splitlines()[-40:]))
print('GATE 2 PASS — license accepted headlessly')


### 📦 6 · Sync project
**What this does:** pulls your Unity project from Drive (or scaffolds a demo project if none found).

In [ ]:
import subprocess, os, glob, json, time
# --- sync project from Drive (or scaffold a demo one) ---
cfg = json.load(open('/content/unitylab_config.json'))
UNITY = open('/content/unity_bin.txt').read().strip()
WORK = '/content/work'
t0 = time.time()
os.makedirs(WORK, exist_ok=True)
proj_candidates = sorted(glob.glob(cfg['DRIVE_PROJECTS'] + '/*/Assets')) if os.path.isdir(cfg['DRIVE_PROJECTS']) else []
if cfg['SYNC_PROJECT'] and proj_candidates:
    src_assets_parent = os.path.dirname(proj_candidates[0])
    name = os.path.basename(src_assets_parent)
    DST = os.path.join(WORK, name)
    r = subprocess.run(['bash', '-lc', 'mkdir -p "%s" && cp -a "%s/." "%s/"' % (WORK, src_assets_parent, DST)],
                       capture_output=True, text=True)
    print('synced project:', name, 'rc:', r.returncode, '(Drive-mount copies are slow for big assets)')
    if r.returncode != 0: print((r.stderr or '')[-1200:])
    assert r.returncode == 0, 'project copy failed'
else:
    name = 'demo'
    DST = os.path.join(WORK, 'demo')
    if not os.path.isdir(DST + '/Assets'):
        r = subprocess.run(['bash', '-lc', '"%s" -batchmode -quit -nographics -createProject "%s" -logFile /content/create.log; echo EXIT_CODE=$?' % (UNITY, DST)],
                           capture_output=True, text=True, timeout=1800)
        okc = [l for l in (r.stdout or '').splitlines() if l.startswith('EXIT_CODE=')]
        logtxt = open('/content/create.log', errors='ignore').read()[-2000:] if os.path.exists('/content/create.log') else '(no create.log — editor never started)'
        print('demo project created:', okc, '| %.1f min' % ((time.time() - t0) / 60))
        assert os.path.isdir(DST + '/Assets'), 'demo create failed: ' + logtxt
os.makedirs(os.path.join(DST, 'Assets', 'Editor'), exist_ok=True)
with open(os.path.join(DST, 'Assets', 'Editor', 'BuildScript.cs'), 'w') as f:
    f.write('''using System.IO; using System.Linq; using UnityEditor; using UnityEditor.SceneManagement; using UnityEngine;
public static class BuildScript {
    [MenuItem("Build/Perform")]
    public static void PerformBuild() {
        var scenes = new string[] { "Assets/Scenes/Sample.unity" };
        if (!File.Exists("Assets/Scenes/Sample.unity")) {
            var scene = EditorSceneManager.NewScene(NewSceneSetup.DefaultGameObjects, NewSceneMode.Single);
            GameObject.CreatePrimitive(PrimitiveType.Cube);
            System.IO.Directory.CreateDirectory("Assets/Scenes");
            EditorSceneManager.SaveScene(scene, scenes[0]);
            Debug.Log("SCENE_CREATED_BY_SCRIPT");
        }
        EditorBuildSettings.scenes = scenes.Select(s => new EditorBuildSettingsScene(s, true)).ToArray();
        AssetDatabase.Refresh(ImportAssetOptions.ForceUpdate);
        AssetDatabase.SaveAssets();
        if (!File.Exists(scenes[0])) throw new System.Exception("SCENE_SAVE_FAILED: " + scenes[0]);
        var target = BuildTarget.WebGL; var outDir = "Builds/WebGL/probe";
        var report = BuildPipeline.BuildPlayer(scenes, outDir, target, BuildOptions.None);
        if (report.summary.result != UnityEditor.Build.Reporting.BuildResult.Succeeded) throw new System.Exception("BUILD_FAILED: " + report.summary.result);
        Debug.Log("BUILD_OK " + report.summary.totalSize + " bytes");
    }
}
''')
DEMO_DIR = os.path.join(DST, 'Assets', 'Demo')
if name == 'demo' and not os.path.exists(os.path.join(DEMO_DIR, 'DemoBoot.cs')):
    os.makedirs(DEMO_DIR, exist_ok=True)
    GAME_BOOT_CS = '''using UnityEngine;
public static class DemoBoot {
    [RuntimeInitializeOnLoadMethod(RuntimeInitializeLoadType.AfterSceneLoad)]
    static void Boot() {
        var stale = GameObject.Find("Cube");
        if (stale != null) Object.Destroy(stale);
        if (GameObject.Find("CubeDodger") != null) return;
        new GameObject("CubeDodger").AddComponent<GameManager>();
        Debug.Log("GAME_BOOTED");
    }
}
'''
    open(os.path.join(DEMO_DIR, 'DemoBoot.cs'), 'w').write(GAME_BOOT_CS)
    GAME_MANAGER_CS = '''using UnityEngine;
using System.Collections.Generic;

public class GameManager : MonoBehaviour {
    public static GameManager Instance { get; private set; }
    Transform ptx; Vector3 p0, cam0;
    List<GameObject> cubes = new List<GameObject>();
    float spawnT = 0.8f, speed = 7f, score, best;
    bool alive = true;
    AudioClip beep;

    void Start() {
        Instance = this;
        var cap = GameObject.CreatePrimitive(PrimitiveType.Capsule);
        cap.name = "Player";
        p0 = new Vector3(0f, 1f, 0f);
        ptx = cap.transform;
        ptx.position = p0;
        cam0 = new Vector3(0f, 6.5f, p0.z - 9f);
        beep = MakeBeep(220f, 0.25f);
        Debug.Log("GAME_READY");
    }

    void Update() {
        if (!alive) {
            if (Input.anyKeyDown || Input.GetMouseButtonDown(0)) Restart();
            return;
        }
        speed = 7f + score * 0.05f;
        Vector3 pos = ptx.position;
        pos.z += speed * Time.deltaTime;
        float mv = Input.GetAxisRaw("Horizontal");
        if (Input.touchCount > 0 && Input.GetTouch(0).phase == TouchPhase.Began)
            mv = (Input.GetTouch(0).position.x >= Screen.width / 2) ? 1f : -1f;
        else if (Input.GetMouseButtonDown(0))
            mv = (Input.mousePosition.x >= Screen.width / 2) ? 1f : -1f;
        pos.x = Mathf.Clamp(pos.x + mv * 9f * Time.deltaTime, -9f, 9f);
        ptx.position = pos;
        score = pos.z;
        spawnT -= Time.deltaTime;
        if (spawnT <= 0) { spawnT = Mathf.Max(0.3f, 1.1f - score * 0.004f); Spawn(pos.z); }
        float fall = 4.5f + score * 0.01f;
        for (int i = cubes.Count - 1; i >= 0; i--) {
            var t = cubes[i].transform;
            t.position += Vector3.down * fall * Time.deltaTime;
            if (t.position.z < pos.z - 12f) { Destroy(cubes[i]); cubes.RemoveAt(i); continue; }
            if (Mathf.Abs(t.position.x - pos.x) < 1.0f && Mathf.Abs(t.position.z - pos.z) < 1.0f && t.position.y < 1.4f && t.position.y > -2f) Die();
        }
        var cam = Camera.main.transform;
        cam.position = new Vector3(pos.x * 0.3f, 6.5f, pos.z - 9f);
        cam.LookAt(pos + Vector3.forward * 4f);
    }

    void Spawn(float z) {
        var g = GameObject.CreatePrimitive(PrimitiveType.Cube);
        g.name = "Hazard";
        g.transform.position = new Vector3(Random.Range(-9f, 9f), 13f, z + Random.Range(45f, 70f));
        cubes.Add(g);
    }

    void Die() {
        if (!alive) return;
        alive = false;
        best = Mathf.Max(best, score);
        if (beep != null) AudioSource.PlayClipAtPoint(beep, new Vector3(0f, 2f, 0f));
        Debug.Log("GAME_DIED " + score.ToString("0") + "m");
    }

    void Restart() {
        alive = true; score = 0; spawnT = 0.8f;
        foreach (var c in cubes) Destroy(c);
        cubes.Clear();
        ptx.position = p0;
        Camera.main.transform.position = cam0;
        Debug.Log("GAME_RESTART");
    }

    void OnGUI() {
        if (alive) {
            GUI.Label(new Rect(10, 8, 320, 30), "Distance: " + score.ToString("0") + " m   (A/D or tap sides)");
        } else {
            GUI.Box(new Rect(Screen.width / 2 - 210, Screen.height / 2 - 90, 420, 180), "");
            var st = new GUIStyle(GUI.skin.label) { alignment = TextAnchor.MiddleCenter, wordWrap = true, fontSize = 18 };
            GUI.Label(new Rect(Screen.width / 2 - 190, Screen.height / 2 - 70, 380, 140),
                "GAME OVER\n\nDistance: " + score.ToString("0") + " m\nBest: " + best.ToString("0") +
                " m\n\nPress any key — or tap — to restart", st);
        }
    }

    AudioClip MakeBeep(float freq, float dur) {
        int f = 44100, n = (int)(f * dur);
        var d = new float[n];
        for (int i = 0; i < n; i++) {
            float env = 1f - (float)i / n;
            d[i] = 0.5f * Mathf.Cos(6.28318f * freq * i / f) * env * env;
        }
        var clip = AudioClip.Create("beep", n, 1, f, false);
        clip.SetData(d, 0);
        return clip;
    }
}
'''
    open(os.path.join(DEMO_DIR, 'GameManager.cs'), 'w').write(GAME_MANAGER_CS)
    print('DEMO GAME WRITTEN: DemoBoot.cs + GameManager.cs ->', DEMO_DIR)
open('/content/work_project.txt', 'w').write(DST)
open('/content/work_name.txt', 'w').write(name)
print('GATE 3 PASS — project ready at', DST)


### 🏗️ 7 · Build + publish
**What this does:** builds for the form target (WebGL/Linux) with timing + RAM stats, then copies artifacts to Drive. Installs the WebGL module automatically if missing.

In [ ]:
import subprocess, os, json, time, glob, re, shutil
# --- ensure WebGL module (fresh VMs only; Drive-cached like the editor) then build + publish --- GATE 4 ---
cfg = json.load(open('/content/unitylab_config.json'))
UNITY = open('/content/unity_bin.txt').read().strip()
if not os.path.exists('/content/work_project.txt'):
    raise AssertionError('cell 6 (project sync) did not complete — run cell 6 first (its error will name the real cause)')
DST = open('/content/work_project.txt').read().strip()
t0 = time.time()

if cfg['BUILD_TARGET'] == 'WebGL' and not os.path.isdir('/content/ue/Editor/Data/PlaybackEngines/WebGLSupport'):
    import hashlib
    def md5ok(path):
        m = hashlib.md5()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 22), b''):
                m.update(chunk)
        return m.hexdigest()
    cache_w = os.path.join(cfg['DRIVE_CACHE'], 'WebGL.tar.xz')
    local_w = '/content/WebGL.tar.xz'

    def have_valid(p):
        return os.path.exists(p) and md5ok(p) == cfg['WEBGL_MD5']

    if not have_valid(local_w) and os.path.exists(local_w):
        print('stale/corrupt local WebGL tarball removed'); os.remove(local_w)
    if not have_valid(local_w) and have_valid(cache_w):
        shutil.copyfile(cache_w, local_w)
        print('WebGL module from Drive cache (md5 OK)')
    if not have_valid(local_w):
        r = subprocess.run(['bash', '-lc', 'aria2c -x8 -d /content -o WebGL.tar.xz "%s"' % cfg['WEBGL_URL']], capture_output=True, text=True, timeout=900)
        print('WebGL module download rc:', r.returncode)
        assert r.returncode == 0, 'WebGL module download failed: ' + (r.stderr or '')[-800:]
        try:
            shutil.copyfile(local_w, cache_w)
            print('stashed WebGL module to Drive cache')
        except Exception as e:
            print('cache stash best-effort failed (fine):', repr(e))
    assert have_valid(local_w), 'WebGL tarball md5 MISMATCH from all sources — delete cached copies and rerun'
    r = subprocess.run(['bash', '-lc', 'tar -xJf /content/WebGL.tar.xz -C /content/ue && echo EXTRACT_OK'], capture_output=True, text=True, timeout=900)
    print('WebGL module extract:', r.returncode, (r.stdout or '').strip()[:60])
    assert os.path.isdir('/content/ue/Editor/Data/PlaybackEngines/WebGLSupport'), 'WebGLSupport missing after extract - delete /content/WebGL.tar.xz (corrupt?) and rerun'

shutil.rmtree(os.path.join(DST, 'Temp'), ignore_errors=True)
print('TEMP_CLEARED (pre-build hygiene)')
NAME = open('/content/work_name.txt').read().strip() if os.path.exists('/content/work_name.txt') else 'demo'
LIB = os.path.join(DST, 'Library')
LIBTGZ = os.path.join(cfg['DRIVE_CACHE'], 'library-' + NAME + '.tar.gz') if cfg['ENABLE_DRIVE_CACHE'] else None
if LIBTGZ and os.path.exists(LIBTGZ) and not os.path.isdir(LIB):
    tr = subprocess.run(['bash', '-lc', 'mkdir -p "%s" && tar -xzf "%s" -C "%s"' % (DST, LIBTGZ, DST)], capture_output=True, text=True, timeout=1800)
    print('Library cache restore rc:', tr.returncode)
    if tr.returncode != 0: print((tr.stderr or '')[-1200:])

CS = os.path.join(DST, 'Assets', 'Editor', 'BuildScript.cs')
cs = open(CS).read()
cs = cs.replace('var target = BuildTarget.WebGL; var outDir = "Builds/WebGL/probe";',
                ('var target = BuildTarget.%s; var outDir = "Builds/%s/probe";' % (cfg['BUILD_TARGET'], cfg['BUILD_TARGET'])))
open(CS, 'w').write(cs)

SAVE_TAG = 'Linux' if cfg['BUILD_TARGET'] == 'Linux' else 'WebGL'
LOG = '/content/build_%s.log' % SAVE_TAG.lower()
TF = '/content/time_%s.txt' % SAVE_TAG.lower()
BUILD_CMD = ('export XDG_RUNTIME_DIR=/tmp/xdg; cd "%s" && /usr/bin/time -v "%s" '
             '-batchmode -nographics -quit -projectPath "%s" -executeMethod BuildScript.PerformBuild '
             '-logFile %s 2> %s; echo EXIT_CODE=$?' % (DST, UNITY, DST, LOG, TF))
try:
    r = subprocess.run(['bash', '-lc', BUILD_CMD], capture_output=True, text=True, timeout=2700)
except Exception as e:
    print('RUNTIME_TIMEOUT_OR_ERROR:', repr(e))
    lg = open(LOG, errors='ignore').read() if os.path.exists(LOG) else '(no log)'
    print('\n'.join(lg.splitlines()[-50:])); raise
print('BASH_RETURN_CODE:', r.returncode)
peak = None
if os.path.exists(TF):
    for line in open(TF, errors='ignore'):
        if line.startswith(('Elapsed (wall clock)', 'Maximum resident', 'Percent of CPU', 'Exit status')):
            print('MEASURE:', line.strip())
    m = re.search(r'Maximum resident set size.*?: (\d+)', open(TF, errors='ignore').read())
    peak = int(m.group(1)) if m else None
log = open(LOG, errors='ignore').read() if os.path.exists(LOG) else '(no log)'
ok = 'BUILD_OK' in log
print('exit:', r.returncode, '| wall: %.1f min' % ((time.time() - t0) / 60), '| peak RAM: %.2f GB' % ((peak or 0) / 1048576.0), '| BUILD_OK:', ok)
out_dir = os.path.join(DST, 'Builds', SAVE_TAG)
arts = sorted(glob.glob(out_dir + '/*'))
print('artifacts:', len(arts), 'entries in', out_dir)
if not ok:
    print('log tail:'); print('\n'.join(log.splitlines()[-50:]))
assert ok, 'BUILD FAILED'

stamp = time.strftime('%Y%m%d-%H%M%S')
dest = os.path.join(cfg['DRIVE_BUILDS'], stamp + '-' + SAVE_TAG)
os.makedirs(dest, exist_ok=True)
r = subprocess.run(['bash', '-lc', 'cp -a "%s" "%s/"' % (out_dir, dest)], capture_output=True, text=True)
print('PUBLISH rc:', r.returncode, '->', dest)
if r.returncode != 0: print((r.stderr or '')[-1200:])
n_pub = len(glob.glob(dest + '/**', recursive=True))
print('PUBLISHED to Drive:', n_pub, 'files at', dest)
if LIB and os.path.isdir(LIB):
    if LIBTGZ:
        try:
            tl = time.time()
            subprocess.run(['bash', '-lc', 'tar -czf "%s" -C "%s" Library' % (LIBTGZ, DST)], timeout=1800)
            print('Library persisted: %.2f GB in %.1f s' % (os.path.getsize(LIBTGZ) / 1073741824.0, time.time() - tl))
        except Exception as e:
            print('Library persist best-effort failed (fine):', repr(e))
print('GATE 4 PASS — %s build' % SAVE_TAG)


### 📋 8 · Report
**What this does:** prints the full system state — paste this cell's output back to the agent.

In [ ]:
import subprocess, os, json, glob
# --- system report — paste this whole output back to the agent ---
cfg = json.load(open('/content/unitylab_config.json'))
print('===== unity-lab v0 SYSTEM REPORT =====')
print('EDITOR:', cfg['VERSION'], '| rev:', cfg['REV'])
ubin = '/content/unity_bin.txt'
print('unity_bin:', open(ubin).read().strip() if os.path.exists(ubin) else '(boot cell did not finish)')
lic = os.path.expanduser('~/.config/unity3d/Unity/licenses/UnityEntitlementLicense.xml')
print('license:', 'ACTIVE' if os.path.exists(lic) else 'MISSING')
wproj = '/content/work_project.txt'
print('project:', open(wproj).read().strip() if os.path.exists(wproj) else '(cell 6 did not finish)')
print('form: cache=%s target=%s' % (cfg['ENABLE_DRIVE_CACHE'], cfg['BUILD_TARGET']))
print('local builds:', sorted(glob.glob('/content/work/*/Builds/*/*')))
print('drive builds:', sorted(glob.glob(cfg['DRIVE_BUILDS'] + '/*'))[-5:])
print('drive cache:', sorted(glob.glob(cfg['DRIVE_CACHE'] + '/*')) if os.path.isdir(cfg['DRIVE_CACHE']) else '(cache off)')
print('DISK:', subprocess.run(['bash', '-lc', 'df -h /content | tail -1'], capture_output=True, text=True).stdout.strip())
